# Exercise 6: Multi-Agent Orchestration

**Level:** Challenge

Real-world AI systems rarely use a single agent. In this exercise, you will build a **multi-agent system** with specialized agents that collaborate: a researcher, a writer, and a reviewer. The reviewer can send work back to the writer if quality is insufficient — creating a feedback loop.

**What you will learn:**
- Designing specialized agents with different system prompts
- Orchestrating agents with LangGraph
- Implementing a feedback loop (reviewer → writer)
- Managing shared state across agents

## 1. Setup

In [ ]:
!pip install langgraph langchain langchain-openai -q

In [ ]:
import os
os.environ["OPENAI_API_KEY"] = "your-key-here"

## 2. Architecture

Our system has three specialized agents:

```
START → [Researcher] → [Writer] → [Reviewer] → END
                           ↑            |
                           └── (revise) ←┘
```

- **Researcher**: Gathers key facts and talking points about a topic
- **Writer**: Composes an article using the research
- **Reviewer**: Evaluates quality; sends back to Writer if below threshold

## 3. Define State and Agents

In [ ]:
from typing import TypedDict, Annotated
import operator
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from pydantic import BaseModel, Field

# Shared state across all agents
class MultiAgentState(TypedDict):
    topic: str
    research: str
    article: str
    review_score: float
    review_feedback: str
    revision_count: int
    revision_history: Annotated[list[str], operator.add]
    status: str  # "researching", "writing", "reviewing", "approved"

# Models
research_model = ChatOpenAI(model="gpt-4o-mini", temperature=0.3)
writer_model = ChatOpenAI(model="gpt-4o-mini", temperature=0.7)
reviewer_model = ChatOpenAI(model="gpt-4o-mini", temperature=0.1)
parser = StrOutputParser()

print("State and models defined.")

## 4. The Researcher Agent

In [ ]:
def researcher_agent(state: MultiAgentState) -> dict:
    """Research agent: gathers key facts and talking points."""
    prompt = ChatPromptTemplate.from_messages([
        ("system",
         "You are an expert researcher. Given a topic, provide:\n"
         "1. Five key facts or statistics\n"
         "2. Three expert perspectives or quotes (you may paraphrase known positions)\n"
         "3. Two counterarguments or challenges\n"
         "Be factual and cite-worthy. Mark anything uncertain as UNVERIFIED."),
        ("human", "Research this topic thoroughly: {topic}")
    ])
    chain = prompt | research_model | parser
    research = chain.invoke({"topic": state["topic"]})
    print(f"  [Researcher] Completed research ({len(research)} chars)")
    return {"research": research, "status": "researching"}

## 5. The Writer Agent

In [ ]:
def writer_agent(state: MultiAgentState) -> dict:
    """Writer agent: composes an article from research."""
    revision_count = state.get("revision_count", 0)
    
    if revision_count == 0:
        # First draft
        prompt = ChatPromptTemplate.from_messages([
            ("system",
             "You are an expert article writer. Write a well-structured article using the "
             "provided research. Include:\n"
             "- A compelling title\n"
             "- An engaging introduction\n"
             "- 2-3 body paragraphs with facts from the research\n"
             "- A conclusion\n"
             "Keep it around 300 words. Be professional but engaging."),
            ("human", "Topic: {topic}\n\nResearch:\n{research}\n\nWrite the article.")
        ])
        chain = prompt | writer_model | parser
        article = chain.invoke({"topic": state["topic"], "research": state["research"]})
    else:
        # Revision based on feedback
        prompt = ChatPromptTemplate.from_messages([
            ("system",
             "You are an expert article writer revising your work. Address ALL the reviewer's "
             "feedback while maintaining the article's strengths. Keep it around 300 words."),
            ("human",
             "Topic: {topic}\n\n"
             "Original research:\n{research}\n\n"
             "Current draft:\n{article}\n\n"
             "Reviewer feedback:\n{feedback}\n\n"
             "Revise the article to address the feedback.")
        ])
        chain = prompt | writer_model | parser
        article = chain.invoke({
            "topic": state["topic"],
            "research": state["research"],
            "article": state["article"],
            "feedback": state["review_feedback"]
        })
    
    label = "first draft" if revision_count == 0 else f"revision {revision_count}"
    print(f"  [Writer] Completed {label} ({len(article.split())} words)")
    return {
        "article": article,
        "status": "writing",
        "revision_history": [f"v{revision_count + 1}: {len(article.split())} words"]
    }

## 6. The Reviewer Agent

In [ ]:
class ReviewResult(BaseModel):
    """Structured review output."""
    score: float = Field(description="Overall quality score from 0.0 to 1.0")
    strengths: list[str] = Field(description="What works well in the article")
    weaknesses: list[str] = Field(description="What needs improvement")
    feedback: str = Field(description="Specific, actionable feedback for the writer")

def reviewer_agent(state: MultiAgentState) -> dict:
    """Reviewer agent: evaluates article quality."""
    evaluator = reviewer_model.with_structured_output(ReviewResult)
    
    prompt = ChatPromptTemplate.from_messages([
        ("system",
         "You are a strict editorial reviewer. Evaluate the article on:\n"
         "1. Accuracy — does it use the research correctly?\n"
         "2. Structure — clear intro, body, conclusion?\n"
         "3. Engagement — is it interesting to read?\n"
         "4. Clarity — is the writing clear and professional?\n\n"
         "Score 0.0-1.0. Be critical — only score above 0.8 for genuinely publication-ready work.\n"
         "Provide specific, actionable feedback."),
        ("human",
         "Topic: {topic}\n\n"
         "Research provided to writer:\n{research}\n\n"
         "Article to review:\n{article}")
    ])
    chain = prompt | evaluator
    review = chain.invoke({
        "topic": state["topic"],
        "research": state["research"],
        "article": state["article"]
    })
    
    revision_count = state.get("revision_count", 0) + 1
    print(f"  [Reviewer] Score: {review.score:.2f} | Strengths: {len(review.strengths)} | Weaknesses: {len(review.weaknesses)}")
    
    return {
        "review_score": review.score,
        "review_feedback": review.feedback,
        "revision_count": revision_count,
        "status": "reviewing"
    }

## 7. Build the Multi-Agent Graph

In [ ]:
from langgraph.graph import StateGraph, START, END

def review_router(state: MultiAgentState) -> str:
    """Route based on review score."""
    if state["review_score"] >= 0.8:
        print(f"  [Router] Score {state['review_score']:.2f} >= 0.8 → APPROVED")
        return "approved"
    if state["revision_count"] >= 3:
        print(f"  [Router] Max revisions reached → APPROVED (best effort)")
        return "approved"
    print(f"  [Router] Score {state['review_score']:.2f} < 0.8 → REVISE")
    return "revise"

def finalize(state: MultiAgentState) -> dict:
    """Final node: mark as approved."""
    print(f"  [Finalize] Article approved after {state['revision_count']} review(s)")
    return {"status": "approved"}

# Build the graph
builder = StateGraph(MultiAgentState)

builder.add_node("researcher", researcher_agent)
builder.add_node("writer", writer_agent)
builder.add_node("reviewer", reviewer_agent)
builder.add_node("finalize", finalize)

# Flow
builder.add_edge(START, "researcher")
builder.add_edge("researcher", "writer")
builder.add_edge("writer", "reviewer")

# Conditional: reviewer → writer (revise) or reviewer → finalize (approved)
builder.add_conditional_edges(
    "reviewer",
    review_router,
    {
        "revise": "writer",
        "approved": "finalize"
    }
)

builder.add_edge("finalize", END)

multi_agent_graph = builder.compile()
print("Multi-agent graph compiled!")

In [ ]:
# Visualize
print(multi_agent_graph.get_graph().draw_mermaid())

## 8. Run the Multi-Agent Pipeline

In [ ]:
print("Running multi-agent pipeline...\n")
print("=" * 60)

result = multi_agent_graph.invoke({
    "topic": "How AI-powered agents are transforming airline customer service",
    "research": "",
    "article": "",
    "review_score": 0.0,
    "review_feedback": "",
    "revision_count": 0,
    "revision_history": [],
    "status": ""
})

print("\n" + "=" * 60)
print("PIPELINE COMPLETE")
print("=" * 60)
print(f"\nStatus: {result['status']}")
print(f"Reviews: {result['revision_count']}")
print(f"Final score: {result['review_score']:.2f}")
print(f"Revision history: {result['revision_history']}")
print(f"\n{'='*60}")
print("FINAL ARTICLE:")
print("=" * 60)
print(result["article"])

## 9. Streaming Multi-Agent Execution

Watch each agent work in real-time.

In [ ]:
print("Streaming multi-agent execution...\n")

for event in multi_agent_graph.stream({
    "topic": "The impact of sustainable aviation fuel on the future of air travel",
    "research": "",
    "article": "",
    "review_score": 0.0,
    "review_feedback": "",
    "revision_count": 0,
    "revision_history": [],
    "status": ""
}):
    for node_name, output in event.items():
        print(f"\n>>> Agent: {node_name}")
        if "article" in output:
            print(f"    Article: {output['article'][:120]}...")
        if "review_score" in output:
            print(f"    Score: {output['review_score']}")
        if "status" in output:
            print(f"    Status: {output['status']}")

---
## YOUR TURN: Exercise A

Extend the multi-agent system by adding a **fourth agent: the Editor**.

The Editor sits between the Reviewer and the final output. After the Reviewer approves (score >= 0.8), the Editor:
1. Polishes the writing style
2. Adds a compelling headline
3. Adds a one-line summary (for social media)

Update the state, add the node, rewire the graph.

In [ ]:
# YOUR TURN: Add the Editor agent

# TODO: Update the state to include editor_output fields

# TODO: Define the editor_agent function

# TODO: Rebuild the graph with the Editor between Reviewer and Finalize

# TODO: Test the full pipeline

---
## YOUR TURN: Exercise B

Build a **completely different multi-agent system** for code review:

- **Developer Agent**: Given a task description, writes Python code
- **Tester Agent**: Generates test cases for the code
- **Reviewer Agent**: Reviews both code and tests, scores quality
- If score < 0.8 → loop back to Developer with feedback

This is an open-ended challenge. Design the state, prompts, and flow yourself.

In [ ]:
# YOUR TURN: Build the code review multi-agent system

# This is a challenge exercise — less scaffolding is provided intentionally.
# Design your own:
# - State schema
# - Agent prompts
# - Graph topology
# - Routing logic

# Tip: Start simple, get it working, then add complexity.

## Key Takeaways

- **Multi-agent systems** decompose complex tasks into specialized roles
- Each agent has its own **system prompt** and can use a different model/temperature
- **Feedback loops** (reviewer → writer) enable self-improvement
- Always include a **max revision guard** to prevent infinite loops
- **Shared state** coordinates work between agents
- Stream execution to observe agent interactions in real-time
- Design agents to be **composable** — you can add/remove/swap agents easily

**Next:** In Exercise 7, you will build a complete end-to-end capstone agent that combines everything you have learned.